# 6 — Vector Store

We know how to chunk text and how to embed it. Now let's put them together.

We'll use **ChromaDB** — a vector database that stores embeddings and lets you search them by similarity.

The flow:
1. Chunk our sample text
2. Embed each chunk
3. Store chunks + embeddings in ChromaDB
4. Query with natural language and see what comes back

In [ ]:
from dotenv import load_dotenv
from mistralai import Mistral
import chromadb
import os

load_dotenv()

client = Mistral(api_key=os.getenv("mistral_api_key"))

## Step 1: Chunk the text

We'll keep it simple — split by paragraphs.

In [ ]:
with open("sample_text.txt", "r") as f:
    text = f.read()

chunks = [p.strip() for p in text.split("\n\n") if p.strip()]

print(f"Number of chunks: {len(chunks)}")
print(f"First chunk: {chunks[0][:100]}...")

## Step 2: Embed the chunks

Send all chunks to the embedding API in one call.

In [ ]:
response = client.embeddings.create(
    model="mistral-embed",
    inputs=chunks,
)

embeddings = [item.embedding for item in response.data]

print(f"Got {len(embeddings)} embeddings, each with {len(embeddings[0])} dimensions")

## Step 3: Store in ChromaDB

ChromaDB needs three things for each entry:
- `documents` — the original text (so we can read what was retrieved)
- `embeddings` — the vectors (for similarity search)
- `ids` — unique identifiers

In [ ]:
chroma_client = chromadb.PersistentClient(path="dbs")

# delete the collection if it already exists (so we can re-run this cell)
try:
    chroma_client.delete_collection(name="chocolate_demo")
except:
    pass

collection = chroma_client.create_collection(name="chocolate_demo")

collection.add(
    documents=chunks,
    embeddings=embeddings,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
)

print(f"Stored {collection.count()} chunks in ChromaDB")

## Step 4: Query it

Now the fun part. Ask a question in plain English and ChromaDB finds the most relevant chunks.

We embed the question with the same model, then search by vector similarity.

In [ ]:
query = "is white chocolate real chocolate?"

query_embedding = client.embeddings.create(
    model="mistral-embed",
    inputs=[query],
).data[0].embedding

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
)

print(f"Query: {query}")
print("---")
for i, doc in enumerate(results["documents"][0]):
    dist = results["distances"][0][i]
    print(f"\nResult {i+1} (distance: {dist:.4f}):")
    print(doc)

Notice how it found the relevant paragraph even though the query and the text use different words. That's semantic search — it matches by **meaning**, not keywords.

## Try different queries

Change the query below and re-run:

In [ ]:
query = "how should I store chocolate?"

query_embedding = client.embeddings.create(
    model="mistral-embed",
    inputs=[query],
).data[0].embedding

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
)

print(f"Query: {query}")
print("---")
for i, doc in enumerate(results["documents"][0]):
    dist = results["distances"][0][i]
    print(f"\nResult {i+1} (distance: {dist:.4f}):")
    print(doc)

## What's the `distance`?

ChromaDB returns distances, not similarities. Lower = more similar.

- The top result should be very relevant
- Results 2 and 3 might be related but less on-topic
- `n_results` controls how many chunks you get back — more results = more context but also more noise

Next up: we feed these retrieved chunks to the LLM to get an actual answer.